# Supplementary Results 9 — Non-linearity of pleiotropy against drug target success

Three nested logistic models per pleiotropy measure — genetic support alone, plus `log(X+1)`, plus
its square — compared by likelihood-ratio test, then 1,000 bootstrap resamples of the quadratic
model, then a sensitivity analysis excluding targets with documented safety liabilities.

Numbers are written to `results/sr09_nonlinearity.json`.

**Provenance.** `chapters/_legacy/02-analysis/06-target-enrichment/11-non-linearity-gPS.ipynb`,
cells 9-19: the same formulas, the same `lr_test`, and the same bootstrap. The published bootstrap
run fixed no seed, so its mean, standard deviation and interval are reproduced to Monte Carlo error
rather than exactly; this notebook fixes a seed so its own numbers are stable.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

from manuscript_methods import paper

numbers = {}
BOOTSTRAP_ITERATIONS = 1000
SEED = 0

pairs = pd.read_csv(paper.derived("df_for_enrichment_regression.csv"))
print(f"target-indication pairs: {len(pairs):,} | approved: {int(pairs['outcome'].sum()):,}")

target-indication pairs: 37,377 | approved: 4,564


## Likelihood-ratio tests for the quadratic term

In [2]:
def models(frame, measure):
    """Baseline, linear-log and quadratic-log models of approval for one pleiotropy measure."""
    linear_term = f"I(np.log({measure}+1))"
    quadratic_term = f"I(np.log({measure}+1) ** 2)"
    return {
        "baseline": smf.logit("outcome ~ geneticSupport", data=frame).fit(disp=False),
        "linear": smf.logit(f"outcome ~ geneticSupport + {linear_term}", data=frame).fit(disp=False),
        "quadratic": smf.logit(f"outcome ~ geneticSupport + {linear_term} + {quadratic_term}", data=frame).fit(
            disp=False
        ),
    }


def lr_test(full, restricted):
    """Likelihood-ratio statistic, degrees of freedom and P value for two nested models."""
    statistic = 2.0 * (float(full.llf) - float(restricted.llf))
    degrees = len(full.params) - len(restricted.params)
    return {"LR": round(statistic, 2), "df": degrees, "P": float(chi2.sf(statistic, degrees))}


fits = {measure: models(pairs, measure) for measure in ["uniqueTherapeuticAreas", "uniqueDiseases"]}
tests = pd.DataFrame(
    [
        {"measure": measure, "comparison": label, **lr_test(fitted[full], fitted[restricted])}
        for measure, fitted in fits.items()
        for label, full, restricted in [
            ("quadratic vs baseline", "quadratic", "baseline"),
            ("linear vs baseline", "linear", "baseline"),
            ("quadratic vs linear", "quadratic", "linear"),
        ]
    ]
)
tests

,measure,comparison,LR,df,P
0,uniqueTherapeuticAreas,quadratic vs baseline,80.54,2,3.244042e-18
1,uniqueTherapeuticAreas,linear vs baseline,15.64,1,7.653228e-05
2,uniqueTherapeuticAreas,quadratic vs linear,64.90,1,7.890355e-16
3,uniqueDiseases,quadratic vs baseline,66.38,2,3.847333e-15
4,uniqueDiseases,linear vs baseline,12.21,1,4.751068e-04
5,uniqueDiseases,quadratic vs linear,54.17,1,1.836974e-13


In [3]:
def test_value(measure, comparison, column):
    """One cell of the likelihood-ratio table."""
    row = tests[(tests["measure"] == measure) & (tests["comparison"] == comparison)].iloc[0]
    return float(row[column])


numbers["S9.01"] = test_value("uniqueTherapeuticAreas", "quadratic vs baseline", "LR")
numbers["S9.02"] = test_value("uniqueTherapeuticAreas", "quadratic vs baseline", "P")
numbers["S9.03"] = test_value("uniqueTherapeuticAreas", "quadratic vs linear", "LR")
numbers["S9.04"] = test_value("uniqueTherapeuticAreas", "quadratic vs linear", "P")
numbers["S9.05"] = test_value("uniqueDiseases", "quadratic vs baseline", "LR")
numbers["S9.06"] = test_value("uniqueDiseases", "quadratic vs baseline", "P")
numbers["S9.07"] = test_value("uniqueDiseases", "quadratic vs linear", "LR")
numbers["S9.08"] = test_value("uniqueDiseases", "quadratic vs linear", "P")
print({k: numbers[k] for k in ["S9.01", "S9.03", "S9.05", "S9.07"]})

{'S9.01': 80.54, 'S9.03': 64.9, 'S9.05': 66.38, 'S9.07': 54.17}


## The fitted curve and its peak

Supplementary Results 11 quotes the peak of the therapeutic-area curve, `exp(-b / 2a) - 1`, as the
reason the combined criterion sits at intermediate pleiotropy.

In [4]:
def terms(measure):
    """Linear and quadratic log coefficients of the quadratic model."""
    fitted = fits[measure]["quadratic"]
    linear = fitted.params[f"I(np.log({measure} + 1))"]
    quadratic = fitted.params[f"I(np.log({measure} + 1) ** 2)"]
    return float(linear), float(quadratic)


ta_linear, ta_quadratic = terms("uniqueTherapeuticAreas")
gps_linear, gps_quadratic = terms("uniqueDiseases")
peak = float(np.exp(-ta_linear / (2 * ta_quadratic)) - 1)

numbers["S9.09"] = round(ta_linear, 3)
numbers["S9.14"] = round(ta_quadratic, 3)
numbers["S9.19"] = round(gps_linear, 3)
numbers["S9.24"] = round(gps_quadratic, 3)
numbers["S9.33"] = round(peak, 2)
print(f"therapeutic areas: linear {ta_linear:.3f}, quadratic {ta_quadratic:.3f}, peak at {peak:.2f} areas")
print(f"unique diseases: linear {gps_linear:.3f}, quadratic {gps_quadratic:.3f}")

therapeutic areas: linear 0.568, quadratic -0.267, peak at 1.90 areas
unique diseases: linear 0.371, quadratic -0.120


## Bootstrap robustness

1,000 resamples with replacement; for each, the quadratic model is refitted and both log terms
recorded. Sign stability and the share of resamples reaching P < 0.05 are reported alongside.

In [5]:
def bootstrap(measure, iterations=BOOTSTRAP_ITERATIONS, seed=SEED):
    """Refit the quadratic model on resampled pairs and collect both log-term coefficients."""
    linear_term = f"I(np.log({measure} + 1))"
    quadratic_term = f"I(np.log({measure} + 1) ** 2)"
    formula = f"outcome ~ geneticSupport + I(np.log({measure}+1)) + I(np.log({measure}+1) ** 2)"
    generator = np.random.default_rng(seed)
    rows = []
    for _ in range(iterations):
        sample = pairs.iloc[generator.integers(0, len(pairs), len(pairs))]
        try:
            fitted = smf.logit(formula, data=sample).fit(disp=False)
        except Exception:
            continue
        rows.append(
            {
                "linear": float(fitted.params[linear_term]),
                "quadratic": float(fitted.params[quadratic_term]),
                "linearP": float(fitted.pvalues[linear_term]),
                "quadraticP": float(fitted.pvalues[quadratic_term]),
            }
        )
    return pd.DataFrame(rows)


draws = {measure: bootstrap(measure) for measure in ["uniqueTherapeuticAreas", "uniqueDiseases"]}
summary = pd.DataFrame(
    [
        {
            "measure": measure,
            "term": term,
            "bootstrap mean": round(float(frame[term].mean()), 3),
            "bootstrap SD": round(float(frame[term].std()), 3),
            "CI low": round(float(frame[term].quantile(0.025)), 3),
            "CI high": round(float(frame[term].quantile(0.975)), 3),
            "sign stability (%)": round(100 * float((np.sign(frame[term]) == np.sign(frame[term].mean())).mean()), 1),
            "P < 0.05 (%)": round(100 * float((frame[f"{term}P"] < 0.05).mean()), 1),
        }
        for measure, frame in draws.items()
        for term in ["linear", "quadratic"]
    ]
)
summary

,measure,term,bootstrap mean,bootstrap SD,CI low,CI high,sign stability (%),P < 0.05 (%)
0,uniqueTherapeuticAreas,linear,0.567,0.064,0.444,0.690,100.0,100.0
1,uniqueTherapeuticAreas,quadratic,-0.266,0.034,-0.333,-0.199,100.0,100.0
2,uniqueDiseases,linear,0.370,0.047,0.279,0.456,100.0,100.0
3,uniqueDiseases,quadratic,-0.120,0.018,-0.154,-0.085,100.0,100.0


In [6]:
def draw_value(measure, term, column):
    """One cell of the bootstrap summary."""
    row = summary[(summary["measure"] == measure) & (summary["term"] == term)].iloc[0]
    return float(row[column])


numbers["S9.10"] = draw_value("uniqueTherapeuticAreas", "linear", "bootstrap mean")
numbers["S9.11"] = draw_value("uniqueTherapeuticAreas", "linear", "bootstrap SD")
numbers["S9.12"] = draw_value("uniqueTherapeuticAreas", "linear", "CI low")
numbers["S9.13"] = draw_value("uniqueTherapeuticAreas", "linear", "CI high")
numbers["S9.15"] = draw_value("uniqueTherapeuticAreas", "quadratic", "bootstrap mean")
numbers["S9.16"] = draw_value("uniqueTherapeuticAreas", "quadratic", "bootstrap SD")
numbers["S9.17"] = draw_value("uniqueTherapeuticAreas", "quadratic", "CI low")
numbers["S9.18"] = draw_value("uniqueTherapeuticAreas", "quadratic", "CI high")
numbers["S9.20"] = draw_value("uniqueDiseases", "linear", "bootstrap mean")
numbers["S9.21"] = draw_value("uniqueDiseases", "linear", "bootstrap SD")
numbers["S9.22"] = draw_value("uniqueDiseases", "linear", "CI low")
numbers["S9.23"] = draw_value("uniqueDiseases", "linear", "CI high")
numbers["S9.25"] = draw_value("uniqueDiseases", "quadratic", "bootstrap mean")
numbers["S9.26"] = draw_value("uniqueDiseases", "quadratic", "bootstrap SD")
numbers["S9.27"] = draw_value("uniqueDiseases", "quadratic", "CI low")
numbers["S9.28"] = draw_value("uniqueDiseases", "quadratic", "CI high")
numbers["S9.29"] = min(draw_value(m, t, "sign stability (%)") for m in draws for t in ["linear", "quadratic"])
numbers["S9.30"] = min(draw_value(m, t, "P < 0.05 (%)") for m in draws for t in ["linear", "quadratic"])
print(f"lowest sign stability {numbers['S9.29']}% | lowest significance rate {numbers['S9.30']}%")

lowest sign stability 100.0% | lowest significance rate 100.0%


## Sensitivity — excluding targets with documented safety liabilities

The manuscript reports the dataset falling from 37,377 to 8,433 pairs. Dropping the three safety
gene sets alone removes far fewer pairs than that, so the count is printed rather than assumed and
both readings are reported.

In [7]:
gene_sets = pd.read_parquet(paper.derived("gene_sets"))
liability_sets = ["trial_safety_concern", "withdrawn_drug", "liable_target"]
liable = set(gene_sets.loc[gene_sets["geneSet"].isin(liability_sets), "geneId"])
annotated_targets = set(gene_sets["geneId"])

without_liabilities = pairs[~pairs["targetId"].isin(liable)]
annotated_without = pairs[pairs["targetId"].isin(annotated_targets) & ~pairs["targetId"].isin(liable)]
print(f"pairs after dropping the three safety sets: {len(without_liabilities):,}")
print(f"pairs whose target carries any gene-set annotation and no liability: {len(annotated_without):,}")

sensitivity = pd.DataFrame(
    [
        {
            "subset": label,
            "pairs": len(frame),
            "quadratic coefficient": round(
                float(
                    smf.logit(
                        "outcome ~ geneticSupport + I(np.log(uniqueTherapeuticAreas+1))"
                        " + I(np.log(uniqueTherapeuticAreas+1) ** 2)",
                        data=frame,
                    )
                    .fit(disp=False)
                    .params["I(np.log(uniqueTherapeuticAreas + 1) ** 2)"]
                ),
                3,
            ),
        }
        for label, frame in [
            ("all pairs", pairs),
            ("dropping the three safety sets", without_liabilities),
            ("annotated targets without a liability", annotated_without),
        ]
    ]
)
numbers["S9.31"] = int(sensitivity.loc[1, "pairs"])
numbers["S9.32"] = float(sensitivity.loc[1, "quadratic coefficient"])
sensitivity

pairs after dropping the three safety sets: 8,433
pairs whose target carries any gene-set annotation and no liability: 7,531


,subset,pairs,quadratic coefficient
0,all pairs,37377,-0.267
1,dropping the three safety sets,8433,-0.474
2,annotated targets without a liability,7531,-0.372


## Write the results

In [8]:
print(paper.save_results("sr09_nonlinearity", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr09_nonlinearity.json


,computed
S9.01,8.054000e+01
S9.02,3.244042e-18
S9.03,6.490000e+01
S9.04,7.890355e-16
S9.05,6.638000e+01
S9.06,3.847333e-15
S9.07,5.417000e+01
S9.08,1.836974e-13
S9.09,5.680000e-01
S9.14,-2.670000e-01
